In [1]:
import os
import requests
import pandas as pd
import time
from dotenv import load_dotenv

# Since the files are in the same folder, this is the cleanest way:
load_dotenv() 

api_key = os.getenv("SERPAPI_KEY")

if api_key:
    print(f"✅ Success! Connection established with key: {api_key[:5]}...")
else:
    print("❌ Error: Still can't see the key. Restart your Jupyter kernel (top menu).")

✅ Success! Connection established with key: dc0d9...


In [ ]:
import serpapi
import pandas as pd
import time

client = serpapi.Client(api_key="dc0d9e7d3afaa859eafa61dab252b9299419c59e8d9c641df712f15088b37dfd")

# Your verified list of 23 Berlin restaurants
berlin_rids = [
    "r/sticksnsushi-potsdamer-berlin", "r/borchardt-berlin", "r/grill-royal-berlin",
    "r/crackers-berlin", "r/facil-berlin", "r/kin-za-berlin",
    "r/block-house-am-alexanderplatz-berlin", "r/893-ryotei-berlin",
    "r/vox-restaurant-berlin", "r/neni-berlin", "r/katz-orange",
    "r/peterpaul-berlin", "r/cafe-wintergarten-berlin", "r/maison-papillon-berlin",
    "r/kink-bar-and-restaurant-berlin", "r/ottenthal-restaurant-and-weinhandlung",
    "r/jules-verne-restaurant", "r/wilde-matilde-berlin",
    "r/beef-grillclub-by-hasir-leipziger-platz", "r/anna-and-paul-berlin",
    "r/miyaki-japan-meets-persia-berlin", "r/the-alchemist-bar-and-restaurant-berlin",
    "r/pizzeria-mario-x-gambino-berlin"
]

all_reviews = []
MAX_PAGES_PER_RES = 5  # 50 reviews per restaurant to hit your 1000 goal quickly

print(f"Target: ~1,100 reviews. Starting collection...")

for rid in berlin_rids:
    print(f"Scraping {rid}...", end=" ")
    count = 0
    for page in range(MAX_PAGES_PER_RES):
        results = client.search({
            "engine": "open_table_reviews",
            "rid": rid,
            "offset": page * 10
        })
        
        reviews = results.get("reviews", [])
        if not reviews:
            break
            
        for r in reviews:
            r['restaurant_id'] = rid
            all_reviews.append(r)
        
        count += len(reviews)
        time.sleep(0.2) # Faster pace to finish tonight
        
    print(f"Done ({count} reviews).")

# 3. Create DataFrame and Save
df = pd.DataFrame(all_reviews)

print(f"\n✅ SUCCESS!")
print(f"Total reviews collected: {len(df)}")
print(f"Credits used: approx. {len(df)//10}")

Target: ~1,100 reviews. Starting collection...
Scraping r/sticksnsushi-potsdamer-berlin... Done (50 reviews).
Scraping r/borchardt-berlin... Done (0 reviews).
Scraping r/grill-royal-berlin... Done (0 reviews).
Scraping r/crackers-berlin... Done (0 reviews).
Scraping r/facil-berlin... Done (0 reviews).
Scraping r/kin-za-berlin... Done (0 reviews).
Scraping r/block-house-am-alexanderplatz-berlin... Done (0 reviews).
Scraping r/893-ryotei-berlin... Done (0 reviews).
Scraping r/vox-restaurant-berlin... Done (50 reviews).
Scraping r/neni-berlin... Done (50 reviews).
Scraping r/katz-orange... Done (50 reviews).
Scraping r/peterpaul-berlin... Done (50 reviews).
Scraping r/cafe-wintergarten-berlin... Done (50 reviews).
Scraping r/maison-papillon-berlin... Done (50 reviews).
Scraping r/kink-bar-and-restaurant-berlin... Done (50 reviews).
Scraping r/ottenthal-restaurant-and-weinhandlung... Done (0 reviews).
Scraping r/jules-verne-restaurant... Done (0 reviews).
Scraping r/wilde-matilde-berlin...

In [ ]:
# New list of popular Berlin RIDs to replace the '0 review' ones
rescue_rids = [
    "r/facil-berlin", "r/pauly-saal-berlin", "r/monsieur-vuong-berlin", 
    "r/vanecek-berlin", "r/lutter-and-wegner-berlin", "r/reinstoff-berlin",
    "r/tim-raue-berlin", "r/crack-pfeffer-berlin", "r/zur-letzten-instanz-berlin",
    "r/brauhaus-lemke-am-schloss-berlin"
]

all_rescue_reviews = []

print(f"Starting Rescue Collection...")

for rid in rescue_rids:
    print(f"Scraping {rid}...", end=" ")
    for page in range(5): # Still sticking to 50 reviews per restaurant
        results = client.search({
            "engine": "open_table_reviews",
            "rid": rid,
            "offset": page * 10
        })
        reviews = results.get("reviews", [])
        if not reviews: break
        for r in reviews:
            r['restaurant_id'] = rid
            all_rescue_reviews.append(r)
        time.sleep(0.2)
    print("Done.")

# Combine with your existing 650 reviews
import pandas as pd
old_df = pd.read_csv("berlin_1000_reviews.csv") # The file from your screenshot
new_df = pd.DataFrame(all_rescue_reviews)
final_df = pd.concat([old_df, new_df], ignore_index=True)

print(f"Total rows: {len(final_df)}")

Starting Rescue Collection...
Scraping r/facil-berlin... Done.
Scraping r/pauly-saal-berlin... Done.
Scraping r/monsieur-vuong-berlin... Done.
Scraping r/vanecek-berlin... Done.
Scraping r/lutter-and-wegner-berlin... Done.
Scraping r/reinstoff-berlin... Done.
Scraping r/tim-raue-berlin... Done.
Scraping r/crack-pfeffer-berlin... Done.
Scraping r/zur-letzten-instanz-berlin... Done.
Scraping r/brauhaus-lemke-am-schloss-berlin... Done.
Total rows: 650


In [4]:
final_df.shape

(650, 10)

In [5]:
df.shape

(650, 10)

In [6]:
import pandas as pd

# 1. Simple concatenation to keep EVERYTHING
# axis=0 means stack them vertically
combined2_df = pd.concat([final_df, df], axis=0, ignore_index=True)

# 2. Verify the final count
print(f"Original Dataset 1: {len(final_df)} rows")
print(f"Original Dataset 2: {len(df)} rows")
print(f"Total Rows in Combined Dataset: {len(combined2_df)}")

# 3. Save the full 1,300 rows to your CSV
combined2_df.to_csv("berlin_full_1300_reviews.csv", index=False)
print("✅ Success! Your complete dataset is saved as 'berlin_full_1300_reviews.csv'")

# 4. Final check of the shape
combined2_df.shape

Original Dataset 1: 650 rows
Original Dataset 2: 650 rows
Total Rows in Combined Dataset: 1300
✅ Success! Your complete dataset is saved as 'berlin_full_1300_reviews.csv'


(1300, 10)

In [7]:
df_raw = pd.read_csv('berlin_full_1300_reviews.csv')

In [9]:
from deep_translator import GoogleTranslator

# Test on 5 rows first
test_sample = df_raw['content'].head(5).tolist()
translated_sample = GoogleTranslator(source='de', target='en').translate_batch(test_sample)

for ger, eng in zip(test_sample, translated_sample):
    print(f"GER: {ger[:50]}...")
    print(f"ENG: {eng[:50]}...")
    print("-" * 30)

GER: 味に関してはご飯が乾燥していて美味しくなかったです。またそのご飯に黒胡麻が多すぎてジャリジャリしてい...
ENG: 味に関してはご飯が乾燥していて美味しくなかったです。またそのご飯に黒胡麻が多すぎてジャリジャリしてい...
------------------------------
GER: tolle raffinierte Gerichte, toller Service und gut...
ENG: great refined dishes, great service and good value...
------------------------------
GER: Ein schöner Abend mit lieben Freunden an einem der...
ENG: A lovely evening with dear friends at one of the b...
------------------------------
GER: Outstanding food, lovely and always excellent serv...
ENG: Outstanding food, lovely and always excellent serv...
------------------------------
GER: Unfortunately we didn’t get to dine there on 1 may...
ENG: Unfortunately we didn’t get to dine there on 1 may...
------------------------------
